In [1]:
import os
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import fates_calibration_library.utils as utils
import fates_calibration_library.parameter_generation as param

In [2]:
def get_ensemble_string(clm_key, parameter, type, oaat_dir, default_file):
    this_param = clm_key[clm_key.parameter_name == parameter]
    this_param_type = this_param[this_param.type == type]
    if len(this_param_type) > 0:
        ens = os.path.join(oaat_dir, f'{this_param_type.ensemble.values[0]}.nc')
    else:
        ens = default_file

    return ens

def get_values(clm_key, parameter, oaat_dir, default_file):
    
    min_file = get_ensemble_string(clm_key, parameter, 'min',
                                   oaat_dir, default_file)
    max_file = get_ensemble_string(clm_key, parameter, 'max',
                                   oaat_dir, default_file)
    
    min_val = xr.open_dataset(min_file)[parameter].values
    max_val = xr.open_dataset(max_file)[parameter].values
    default_val = xr.open_dataset(default_file)[parameter].values

    return min_val, max_val, default_val


def get_pct_change(minval, maxval, defval):
    min_diff = (defval - minval)/defval
    max_diff = (maxval - defval)/defval

    return min_diff, max_diff

def get_fates_vals(defval, mindiff, maxdiff):
    minval = defval - defval*mindiff
    maxval = defval + defval*maxdiff

    return minval, maxval

def calc_vcmax(slatop, leafcn, flnr, fnr, act25, fnitr):
    
    lnc = 1.0 / (slatop * leafcn)
    vcmax25top = lnc * flnr * fnr * act25 * fnitr
    
    return vcmax25top

def get_target_vcmax(fates_ids, fates_vcmax):
    targets = []
    for id in fates_ids:
        targets.append(fates_vcmax[id])
    target = np.mean(targets)

    return target

def get_target_fnlr(pft, fates_vcmax, slatop, leafcn, act25, fnr, fnitr):
    
    fates_ids = clm_pfts[clm_names[pft]]
    target = get_target_vcmax(fates_ids, fates_vcmax)

    lnc = 1.0/(slatop[pft] * leafcn[pft])

    flnr = target/(lnc * fnr * act25 * fnitr[pft])

    return flnr

In [3]:
param_dir = '/glade/work/afoster/FATES_calibration/parameter_files'
fates_param_file = os.path.join(param_dir, 'fates_params_default_sci.1.81.1_api.38.0.0_crops_vai.nc')
clm_param_file = os.path.join(param_dir, 'ctsm60_params.c241017.nc')

fates_param = xr.open_dataset(fates_param_file)
clm_param = xr.open_dataset(clm_param_file)

clm_key = pd.read_csv(os.path.join(param_dir, 'clm6sp_oaat_key.csv'), header=None)
clm_key.columns = ['ensemble', 'parameter_name', 'type']

oaat_dir = os.path.join(param_dir, 'clm_oaat')

param_list_name = "param_list_sci.1.85.1_api.40.0.0_updates.xls"
param_list_file = os.path.join(param_dir, param_list_name)
param_dat = param.get_param_dictionary(param_list_file)

In [4]:
vcmax_info = param_dat['leaf_vcmax25top']
vcmax_min = vcmax_info['param_min'].values
vcmax_max = vcmax_info['param_max'].values

vcmax_min = np.insert(vcmax_min, 0, 0.0, axis=0)
vcmax_max = np.insert(vcmax_max, 0, 0.0, axis=0)

In [5]:
clm_pft_config = '/glade/work/afoster/FATES_calibration/fates_calibration_library/configs/clm_fates_index.yaml'
clm_pfts = utils.get_config_file(clm_pft_config)

clm_names_config = '/glade/work/afoster/FATES_calibration/fates_calibration_library/configs/clm_index_to_name.yaml'
clm_names = utils.get_config_file(clm_names_config)

In [6]:
slatop = clm_param['slatop'].values
leafcn = clm_param['leafcn'].values
act25 = clm_param['act25'].values
fnr = clm_param['fnr'].values
fnitr = clm_param['fnitr'].values

In [7]:
pfts = np.arange(0, 17)
flnr_mins = []
for pft in pfts:
    flnr_mins.append(get_target_fnlr(pft, vcmax_min, slatop,
                                     leafcn, act25, fnr, fnitr))

flnr_maxes = []
for pft in pfts:
    flnr_maxes.append(get_target_fnlr(pft, vcmax_max, slatop,
                                     leafcn, act25, fnr, fnitr))

/glade/derecho/scratch/afoster/tmp/ipykernel_107643/2671377779.py:57: RuntimeWarning: divide by zero encountered in scalar divide
  lnc = 1.0/(slatop[pft] * leafcn[pft])
/glade/derecho/scratch/afoster/tmp/ipykernel_107643/2671377779.py:59: RuntimeWarning: invalid value encountered in scalar multiply
  flnr = target/(lnc * fnr * act25 * fnitr[pft])


In [8]:
flnr_mins[0] = 0.0
flnr_maxes[0] = 0.0

In [9]:
pft = 3
fates_ids = clm_pfts[clm_names[pft]]
target = get_target_vcmax(fates_ids, vcmax_max)
result = calc_vcmax(slatop[pft], leafcn[pft], flnr_maxes[pft], fnr, act25, fnitr[pft])
print(target, result, f'for pft {pft}')

105.45 105.44999999999999 for pft 3


In [10]:
new_param = clm_param.copy(deep=False)
new_param['flnr'].values[:17] = flnr_mins[:]
new_param.to_netcdf('/glade/work/afoster/FATES_calibration/parameter_files/clm_oaat/CLM6SPoaat0409.nc')

In [11]:
new_param2 = clm_param.copy(deep=False)
new_param2['flnr'].values[:17] = flnr_maxes[:]
new_param2.to_netcdf('/glade/work/afoster/FATES_calibration/parameter_files/clm_oaat/CLM6SPoaat0410.nc')